# PSD Regularization

A covariance matrix must be positive semi-definite, and a correlation matrix must also have ones on its diagonal. Several high-frequency estimators can break these properties, most often the realized kernel of [Barndorff-Nielsen, Hansen, Lunde and Shephard (2011)](https://doi.org/10.1016/j.jeconom.2010.07.009). This tutorial looks at the raw estimates and repairs one with the four methods of `regularise`: the nearest correlation matrix of [Higham (2002)](https://doi.org/10.1093/imanum/22.3.329), the nearest positive semi-definite matrix, eigenvalue cleaning after [Laloux, Cizeau, Bouchaud and Potters (1999)](https://doi.org/10.1103/PhysRevLett.83.1467), and shrinkage towards the identity.

In [ ]:
Pollis.packages("HighFrequencyCovariance", "Plots")

## The Simulated Assets

Four assets with asynchronous ticks, noise and a known true covariance, as in the first tutorial.

In [ ]:
using HighFrequencyCovariance, LinearAlgebra, Random, Statistics
import Plots

ts, true_covar, true_noise, _ = generate_random_path(4, 10000; rng = MersenneTwister(2))  # four assets, 10,000 ticks in all, known true covariance
assets = get_assets(ts)

## Raw Estimates

`estimate_covariance` regularises by default. `regularisation = missing` returns the raw estimate, so its smallest eigenvalue shows whether it is valid. A negative eigenvalue gives a portfolio with a negative variance.

In [ ]:
estimators = [:simple_covariance, :preaveraged_covariance, :two_scales_covariance, :bnhls_covariance]
raw = Dict(m => estimate_covariance(ts, assets, m; regularisation = missing) for m in estimators)
[(method = m, valid_correlation = valid_correlation_matrix(raw[m].correlation), smallest_eigenvalue = round(minimum(eigvals(raw[m].correlation)), sigdigits = 3)) for m in estimators]

## Four Ways to Repair the Realized Kernel

`regularise(est, ts, method)` returns a new `CovarianceMatrix`. Each method's result is checked for validity and compared with the raw estimate, by the squared Frobenius distance between correlation matrices, and with the truth.

In [ ]:
rk = raw[:bnhls_covariance]
fixes = [:nearest_correlation_matrix, :nearest_psd_matrix, :eigenvalue_clean, :identity_regularisation]
fixed = Dict(f => regularise(rk, ts, f) for f in fixes)
[(method = f, valid_correlation = valid_correlation_matrix(fixed[f].correlation), moved = round(squared_frobenius_distance(Matrix(fixed[f].correlation), Matrix(rk.correlation)), sigdigits = 3), correlation_error = round(calculate_mean_abs_distance(fixed[f], true_covar).Correlation_error, sigdigits = 3)) for f in fixes]

## Eigenvalues Before and After

The nearest correlation matrix and the nearest positive semi-definite matrix raise a negative eigenvalue to zero and lower the others, keeping ones on the diagonal. Eigenvalue cleaning changes the eigenvalues that random matrix theory treats as noise. Shrinkage towards the identity moves every eigenvalue part of the way to one, by an amount the package chooses, which need not be enough to make the matrix valid.

In [ ]:
ev(m) = sort(eigvals(Matrix(m.correlation)))
Plots.plot(1:4, ev(rk); marker = :circle, label = "raw kernel", xlabel = "eigenvalue, smallest first", ylabel = "value", title = "Eigenvalues of the correlation matrix")
for f in fixes
	Plots.plot!(1:4, ev(fixed[f]); marker = :circle, label = string(f))
end
Plots.hline!([0.0]; linestyle = :dash, color = :black, label = "")

## Regularising Within the Estimate

The repair can also be asked for in one call, with `regularisation` naming a method, and `only_regulise_if_not_PSD = true` leaves a valid estimate unchanged.

In [ ]:
est = estimate_covariance(ts, assets, :bnhls_covariance; regularisation = :nearest_correlation_matrix, only_regulise_if_not_PSD = true)
(valid_correlation = valid_correlation_matrix(est.correlation), correlation_error = calculate_mean_abs_distance(est, true_covar).Correlation_error)

## Things to Try

- Set the kernel bandwidth `H` lower and higher, and see when the raw kernel stops being valid.
- Use **Raw Against Regularised** in High-Frequency Covariance Matrix Estimation on your own data.